## Brackets and a trailing stop

A Strategy decides at each close with the verbs a venue gives a trader. This one enters on a
break of a channel, either way, and rests an OCO bracket for each entry: a stop and a
take-profit, the first to fill cancelling the other. While the position is open the stop
is trailed behind the close with `replace`, which moves the one order rather than resting a
second. Bitcoin's perpetual at Bybit, four-hour bars of 2025.

In [1]:
import pathlib

import exchange_router
import pandas
import probatum

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}")

probatum 3.0.0, exchange-router 3.3.0


### The bars

Bybit's hourly perpetuals with their funding, fetched once into `bars/`, Bitcoin's
resampled to four hours; the funding events keep their own times.

In [2]:
PATH = pathlib.Path("../bars/bybit-linear-btcusdt-ethusdt-1h-2024-2025")

if PATH.exists():
    perp = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["bybit"]) as router:
        perp = probatum.Data.from_router(router, exchange="bybit", market="linear",
                                         interval="1h", symbols=["BTCUSDT", "ETHUSDT"],
                                         start="2024-01-01", end="2025-12-31", funding=True)
    perp.save(path=PATH)

bars = perp["BTCUSDT"].resample(interval="4h").after(when="2025-01-01")

bars

Data(BTCUSDT, 4h, 2190 bars from 2025-01-01 00:00 to 2025-12-31 20:00)

### The strategy

`init` computes every array once, over all the bars. `next` places orders that fill on a
later bar, never the one it saw: half the account on a break of the 60-bar channel, a stop
two average true ranges away and a take-profit at four. The stop then follows the close at
three, only ever closer. A `Recorder` keeps each bar's stop as the level the next bar is
tested against, and `marks()` hands the chart what the strategy decided on.

In [3]:
class Breakout(probatum.Strategy):
    def __init__(self, length=60, stop=2.0, target=4.0, trail=3.0):
        self.length, self.stop, self.target, self.trail = length, stop, target, trail

    def init(self, data):
        self.upper  = probatum.ta.shift(probatum.ta.highest(data.high, length=self.length))
        self.lower  = probatum.ta.shift(probatum.ta.lowest(data.low, length=self.length))
        self.atr    = probatum.ta.atr(data.high, low=data.low, close=data.close, length=14)
        self.log    = probatum.chart.marks.Recorder(data)
        self.exits  = None
        self.warmup = self.length + 1

    def next(self, bar, account):
        atr = self.atr[bar.i]

        if account.position == 0.0:
            size = account.equity * 0.5 / bar.close
            if bar.close > self.upper[bar.i]:
                account.buy(size=size)
                self.exits = account.bracket(stop=bar.close - self.stop * atr,
                                             take_profit=bar.close + self.target * atr)
            elif bar.close < self.lower[bar.i]:
                account.sell(size=size)
                self.exits = account.bracket(stop=bar.close + self.stop * atr,
                                             take_profit=bar.close - self.target * atr)
        else:
            stop = self.exits.stop
            if account.position > 0.0:
                trailed = max(stop.trigger, bar.close - self.trail * atr)
            else:
                trailed = min(stop.trigger, bar.close + self.trail * atr)
            if trailed != stop.trigger:
                account.replace(stop, trigger=trailed)
            self.log.at_next(bar, stop=trailed)

    def marks(self):
        return [probatum.chart.marks.Line(self.upper, name=f"high {self.length}", width=1),
                probatum.chart.marks.Line(self.lower, name=f"low {self.length}", width=1),
                probatum.chart.marks.Line(self.log["stop"], name="stop", style="dotted")]


result = probatum.backtest(Breakout(), data=bars, venue="bybit_perp")

result

,run,buy_and_hold,low,high,level,method,why_none
total_return_pct,-11.94,-12.38,-29.76,10.18,0.95,"studentized stationary block bootstrap, blocks of 20.6 bars",
cagr_pct,-11.94,-12.38,-29.77,10.18,0.95,"studentized stationary block bootstrap, blocks of 20.6 bars",
volatility_pct,12.42,45.21,10.62,14.90,0.95,"studentized stationary block bootstrap, blocks of 20.6 bars",
sharpe,-0.96,-0.07,-2.92,1.00,0.95,"delta method, Newey-West lag 19, floored at lag zero",
max_drawdown_pct,15.79,36.00,8.39,31.87,0.95,"middle of resampled histories, stationary block bootstrap, blocks of 20.6 bars",
exposure_pct,31.02,,,,,,
num_trades,49,,,,,,
win_rate,0.31,,,,,,1
profit_factor,0.68,,,,,,1
costs,264,,,,,,


### How each round trip ended

`reason` says what closed a round trip: a bracket's stop or take-profit here, since the
strategy places no other exit.

In [4]:
trades = result.trades.to_pandas()
trades["return, %"] = trades["ret"] * 100

trades.groupby(by=["side", "reason"]).agg(trips=("pnl", "size"), pnl=("pnl", "sum"),
                                          mean_return=("return, %", "mean")).round(decimals=2)

trips      pnl  mean_return
side  reason                                  
long  stop            15 -1245.74        -1.83
      take_profit      8  1326.58         3.78
short stop            21 -2458.41        -2.58
      take_profit      5  1184.03         5.39

### The chart

The channel, the stop as it was trailed, and an arrow at each end of every round trip.

In [5]:
result.chart(result.policy.marks(), focus=6 * 60,
             title="Channel breakout with a trailed bracket, BTCUSDT perpetual 4h, 2025")

Chart(bars=2190, panels=3)

### One bracket's life

The order log holds every event of every order. A take-profit that filled, then its
order's life told in words: placed with no side, armed by the entry's fill, and filled.

In [6]:
fills = result.fills
taken = fills[fills.reason == "take_profit"][0]

print(result.explain(taken))

orders = result.orders.to_pandas()
first  = int(orders.index[orders["order"] == taken["order"]][0])

print(result.explain(result.orders[first]))

fill 11, BTCUSDT buy 0.05 at 86,790.20, 2025-02-25 12:00
  reference        the limit's price               86,790.20
  spread           none for a maker                    +0.00
  slippage         none for a maker                    +0.00
  impact           none for a maker                    +0.00
  held in the bar  the bar's high was not reached      +0.00
  fee              maker 0.02% on 4,339.51              0.87
order 18, BTCUSDT take_profit at 86,790.20
  2025-02-24 20:00  placed  take_profit at 86,790.20
  2025-02-25 00:00  armed   take_profit, buy at 86,790.20
  2025-02-25 12:00  filled  0.05 at 86,790.20, fill 11
